# LangChain Agent Middleware Architecture

**Middleware** in LangChain provides a modular, interceptor-based framework to inspect, modify, limit, and safeguard agent operations at key lifecycle hooks.

Instead of embedding boilerplate safety, logging, or memory management directly into tools or prompts, middleware sits around the agent execution loop.

---

### 🎨 Agent Middleware Lifecycle Architecture

![LangChain Agent Middleware Architecture](../images/middleware_diagram.png)

---

### What We Cover in this Notebook:
1. **`SummarizationMiddleware`**: Automatic memory compaction when conversation message count exceeds thresholds.
2. **`HumanInTheLoopMiddleware`**: Pausing critical tool executions to request human approval (`approve`, `edit`, `reject`).
3. **`ModelCallLimitMiddleware`**: Guarding against infinite agent loops and excessive LLM token burn.
4. **Custom `AgentMiddleware`**: Building an observability & response latency tracker via `wrap_model_call`.


---
## 0. Environment Setup & Free Model Initialization

We use **Groq** (`qwen/qwen3.8-27b`) as our ultra-fast, free-tier LLM.

In [23]:
import os
from dotenv import find_dotenv, load_dotenv
from langchain_groq import ChatGroq

# Load environment variables (.env)
load_dotenv(find_dotenv())

# Initialize our free Groq model
free_model = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0.7,
)
print(f"✅ Groq Model initialized successfully: {free_model.model_name}")

✅ Groq Model initialized successfully: qwen/qwen3.8-27b


from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

# Define tool functions with execution logging
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email to a recipient."""
    # This print statement proves exactly WHEN and TO WHOM the tool actually executes!
    print(f"\n📧 [ACTUAL TOOL EXECUTION] Email physically sent to: '{recipient}' | Subject: '{subject}'")
    return f"Email sent to {recipient} with subject '{subject}' and body: '{body}'"

# System prompt instructing the agent that human supervisors may modify actions
hitl_system_prompt = (
    "You are an AI assistant operating under Human-In-The-Loop supervision. "
    "A human supervisor reviews and may edit your tool calls before execution. "
    "If a tool result shows different arguments than originally requested (e.g. an edited recipient), "
    "it means the human supervisor intentionally modified it. Accept the tool result as authoritative "
    "and confirm what was dispatched without trying to re-send."
)

# Build agent with HumanInTheLoopMiddleware
hitl_agent = create_agent(
    model=free_model,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    system_prompt=hitl_system_prompt,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {"allowed_decisions": ["approve", "edit", "reject"]},
                "read_email_tool": False,  # Read operations are safe and never paused
            }
        )
    ],
)
print("✅ HITL Agent compiled with interrupt_on configured for 'send_email_tool'")


In [24]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

# Create agent configured with SummarizationMiddleware
summary_agent = create_agent(
    model=free_model,
    checkpointer=InMemorySaver(),  # Preserves thread history across invoke() turns
    middleware=[
        SummarizationMiddleware(
            model=free_model,
            trigger=("messages", 10),  # Triggers when total messages reach 10
            keep=("messages", 4),       # Keeps the 4 most recent messages
        )
    ],
)

config = {"configurable": {"thread_id": "math-session-1"}}

questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = summary_agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    msg_count = len(response['messages']) # response store all the peviouse conversations so when no. of response reach to 10 it will sumerized.
    print(f"Q: {q} | Messages response count : {msg_count} | Response: {response['messages']}")

print("\n✅ Notice: After reaching 10 messages, SummarizationMiddleware triggered, compressing older turns and resetting active messages to 6!")

Q: What is 2+2? | Messages response count : 2 | Response: [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='8a83e4e3-3e24-400a-a7b5-0dff97ef2bce'), AIMessage(content='2 + 2 = **4**', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 19, 'total_tokens': 28, 'completion_time': 0.01723723, 'completion_tokens_details': None, 'prompt_time': 0.001062586, 'prompt_tokens_details': None, 'queue_time': 0.044228374, 'total_time': 0.018299816}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a124af-1a8f-75b2-9735-75e05291af5e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 19, 'output_tokens': 9, 'total_tokens': 28})]
Q: What is 10*5? | Messages response count : 4 | Response: [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={

---
## 2. Human-In-The-Loop Middleware (HITL)

When autonomous agents have access to side-effect tools (e.g., sending emails, making financial transfers, deleting records), you must keep a human in control.

`HumanInTheLoopMiddleware` interrupts the agent graph right before tool execution, allowing the user to provide 3 distinct responses:
1. **`approve`**: Execute the tool call as generated.
2. **`edit`**: Modify tool arguments before execution.
3. **`reject`**: Abort tool execution and return feedback.

### Scenario 2B: Human Edits Action Parameters (`type: 'edit'`)

Before sending, the human reviewer intercepts and updates the recipient email, subject, or message body.

> **💡 Why did the tool only send once, and why did the LLM comment on the recipient?**
> 1. **The email was NEVER sent before approval**: The middleware paused execution at the interrupt boundary. The tool only executed **once** when resumed with `Command`, sending directly to `lead@company.com`.
> 2. **Goal-Discrepancy Phenomenon**: The initial user message asked for `manager@company.com`, but the human supervisor secretly edited the tool to `lead@company.com`. Without a system prompt guiding it, the LLM notices that the tool result differs from the user's initial prompt and might wonder if a mistake occurred! Our `hitl_system_prompt` instructs the LLM that human supervisors have final authority.


config_edit = {"configurable": {"thread_id": "email-session-edit"}}

print("--- Step 1: Prompting agent on thread 'email-session-edit' ---")
hitl_agent.invoke(
    {"messages": [HumanMessage(content="Send an email to manager@company.com with subject 'Status' and body 'Completed all tasks.')")]},
    config=config_edit,
)
print("Agent state interrupted for approval! (Tool has NOT run yet)")

# Resume with edited parameters
print("\n--- Step 2: Human edits tool arguments and resumes ---")
edited_result = hitl_agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "send_email_tool",
                        "args": {
                            "recipient": "lead@company.com",            # Modified recipient
                            "subject": "Updated Task Status",          # Modified subject
                            "body": "All high-priority tasks completed."  # Modified body
                        },
                    },
                }
            ]
        }
    ),
    config=config_edit,
)
print("\nFinal Response:")
print(edited_result["messages"][-1].content)


In [25]:
config_approve = {"configurable": {"thread_id": "email-session-approve"}}

print("--- Step 1: Prompting agent on thread 'email-session-approve' ---")
hitl_agent.invoke(
    {"messages": [HumanMessage(content="Send an email to manager@company.com with subject 'Status' and body 'Completed all tasks.')")]},
    config=config_approve,
)
print("Agent state interrupted for approval!")

# Resume with approval
print("\n--- Step 2: Human approves the action ---")
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config_approve,
)
print("Final Response:")
print(approved_result["messages"][-1].content)

--- Step 1: Prompting agent on thread 'email-session-approve' ---
Agent state interrupted for approval!

--- Step 2: Human approves the action ---
Final Response:
The email has been sent successfully to manager@company.com with the subject "Status" and body "Completed all tasks."


### Scenario 2B: Human Edits Action Parameters (`type: 'edit'`)

Before sending, the human reviewer intercepts and updates the recipient email, subject, or message body.

In [26]:
config_edit = {"configurable": {"thread_id": "email-session-edit"}}

print("--- Step 1: Prompting agent on thread 'email-session-edit' ---")
hitl_agent.invoke(
    {"messages": [HumanMessage(content="Send an email to manager@company.com with subject 'Status' and body 'Completed all tasks.')")]},
    config=config_edit,
)
print("Agent state interrupted for approval!")

# Resume with edited parameters
print("\n--- Step 2: Human edits tool arguments ---")
edited_result = hitl_agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "send_email_tool",
                        "args": {
                            "recipient": "lead@company.com",            # Modified recipient
                            "subject": "Updated Task Status",          # Modified subject
                            "body": "All high-priority tasks completed."  # Modified body
                        },
                    },
                }
            ]
        }
    ),
    config=config_edit,
)
print("Final Response:")
print(edited_result["messages"][-1].content)

--- Step 1: Prompting agent on thread 'email-session-edit' ---
Agent state interrupted for approval!

--- Step 2: Human edits tool arguments ---
Final Response:
I apologize for the confusion — I keep sending to the wrong recipient. Let me send the correct email now with exactly what you requested.




### Scenario 2C: Human Rejects Action (`type: 'reject'`)

The reviewer aborts the action and supplies an explanation message. The agent handles the feedback gracefully.

In [27]:
config_reject = {"configurable": {"thread_id": "email-session-reject"}}

print("--- Step 1: Prompting agent on thread 'email-session-reject' ---")
hitl_agent.invoke(
    {"messages": [HumanMessage(content="Send an email to manager@company.com with subject 'Status' and body 'Completed all tasks.')")]},
    config=config_reject,
)
print("Agent state interrupted for approval!")

# Resume with rejection message
print("\n--- Step 2: Human rejects the action ---")
rejected_result = hitl_agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                    "message": "Action cancelled: Recipient email address is incorrect.",
                }
            ]
        }
    ),
    config=config_reject,
)
print("Final Response:")
print(rejected_result["messages"][-1].content)

--- Step 1: Prompting agent on thread 'email-session-reject' ---
Agent state interrupted for approval!

--- Step 2: Human rejects the action ---
Final Response:
The send was rejected again with "Recipient email address is incorrect." 

I'm using `manager@company.com` exactly as you provided, so the issue may be with that address itself (e.g., it may not be a valid/recognized recipient in this environment). Could you double-check the correct email address you'd like this sent to?


---
## 3. ModelCallLimitMiddleware: Runaway Protection & Rate Limiting

In production, autonomous agents can accidentally enter repetitive loops. `ModelCallLimitMiddleware` sets strict bounds on how many times the model can be invoked in a single run.

In [28]:
from langchain.agents.middleware import ModelCallLimitMiddleware

# Enforce maximum 1 model call per execution run
safe_limit_agent = create_agent(
    model=free_model,
    middleware=[
        ModelCallLimitMiddleware(run_limit=1, exit_behavior="end")
    ]
)

result = safe_limit_agent.invoke({"messages": [{"role": "user", "content": "What is 7 * 7?"}]})
print("✅ Agent response within run_limit=1:")
print(result["messages"][-1].content)

✅ Agent response within run_limit=1:
$7 \times 7 = 49$


---
## 4. Custom Middleware: Observability & Latency Benchmarking

You can easily write custom middleware by subclassing `AgentMiddleware` and overriding lifecycle methods like `wrap_model_call` or `wrap_tool_call`.

Here, we create an **ObservabilityMiddleware** that measures execution time and logs model call metadata.

In [29]:
import time
from langchain.agents.middleware import AgentMiddleware

class ObservabilityMiddleware(AgentMiddleware):
    """Custom middleware to monitor latency and response telemetry."""
    def wrap_model_call(self, request, handler):
        print("⏱️ [Observability] Model request initiated...")
        start_time = time.time()
        
        # Execute the actual model invocation
        response = handler(request)
        
        elapsed = time.time() - start_time
        ai_msg = response.result[0]
        print(f"✅ [Observability] Model responded in {elapsed:.2f} seconds | Output tokens: {len(ai_msg.content.split())}")
        return response

# Instantiate agent with custom observability middleware
obs_agent = create_agent(
    model=free_model,
    middleware=[ObservabilityMiddleware()]
)

res = obs_agent.invoke({"messages": [{"role": "user", "content": "What is 12 + 12?"}]})
print(f"\nAgent Response: {res['messages'][-1].content}")

⏱️ [Observability] Model request initiated...
✅ [Observability] Model responded in 0.14 seconds | Output tokens: 5

Agent Response: $12 + 12 = 24$


---
## 5. Summary & Middleware Cheatsheet

| Middleware | Lifecycle Hook | Primary Use Case |
| :--- | :--- | :--- |
| **`SummarizationMiddleware`** | `after_model` | Long-term memory compression, prevents context window overflow |
| **`HumanInTheLoopMiddleware`** | `wrap_tool_call` | Safety approvals, parameter edits, and execution vetoes on dangerous tools |
| **`ModelCallLimitMiddleware`** | `wrap_model_call` | Runaway loop prevention and API token cost caps |
| **`ToolCallLimitMiddleware`** | `wrap_tool_call` | Tool invocation rate-limiting |
| **`PIIMiddleware`** | `before_agent` / `after_agent` | Data redaction, masking, and privacy compliance |
| **Custom `AgentMiddleware`** | `wrap_model_call` / `wrap_tool_call` | Telemetry, caching, retries, and corporate auditing |